In [ ]:
%reload_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src" / "global_settings.py").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)
from src.global_settings import settings


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pandas as pd
from tqdm import tqdm

from global_settings import settings

In [ ]:
classifier_ckpt = settings.lm_weights_path

model = AutoModelForSequenceClassification.from_pretrained(classifier_ckpt)
tokenizer = AutoTokenizer.from_pretrained(classifier_ckpt)

In [ ]:
text_dataset_path = settings.text_dataset_path

df = pd.read_csv(text_dataset_path)
df.head()

In [ ]:
preds = []
texts = df["text"].tolist()

max_length = 1024

model.to(torch.device("cuda"))
model.eval()

for text in tqdm(texts):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
    inputs = {k: v.to(torch.device("cuda")) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        pred = torch.argmax(logits, dim=-1).item()
        preds.append(pred)

df["pred"] = preds
df.head()

In [ ]:
df.to_csv(str(text_dataset_path).replace(".csv", "_preds.csv"), index=False)